# Notebook 02 — Preprocessing and Chronological Data Split
**Project:** Design and Evaluation of a Governed Multi-Signal Drift-Aware Intervention Policy under Delayed Labels: A Google Cloud MLOps Prototype 

**Author:** Sahar Imani  
**Notebook:** 2 of 5


This notebook takes the modelling dataset from Notebook 01 and gets it ready for training. It assigns a role to every column, cleans the predictors, handles missing values, and then splits the data by time: 2013 as the historical base period and 2014–2016 as the out-of-time simulation stream. The split is chronological so that nothing from the later period affects the model or the preprocessing applied to it.

## 1. Loading the Prepared Modelling Dataset

In [2]:
# Core libraries for data handling and plots
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

In [3]:
# Load the filtered modelling dataset created in Notebook 1
df_main = pd.read_csv("../data/interim/loan_main_target.csv", low_memory=False)

# Check the size of the dataset
print("Rows:",df_main.shape[0])
print("Columns:",df_main.shape[1])

df_main.head()

Rows: 1176374
Columns: 144


,Unnamed: 0,id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,...,payment_plan_start_date,hardship_length,hardship_dpd,hardship_loan_status,orig_projected_additional_accrued_interest,hardship_payoff_balance_amount,hardship_last_payment_amount,debt_settlement_flag,issue_year,target_bad_loan
0,0,10129454,12000.0,12000.0,12000.0,36 months,10.99%,392.81,B,B2,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,N,2013,0
1,1,10149488,4800.0,4800.0,4800.0,36 months,10.99%,157.13,B,B2,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,N,2013,0
2,2,10149342,27050.0,27050.0,27050.0,36 months,10.99%,885.46,B,B2,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,N,2013,0
3,3,10148122,12000.0,12000.0,12000.0,36 months,7.62%,373.94,A,A3,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,N,2013,0
4,4,10129477,14000.0,14000.0,14000.0,36 months,12.85%,470.71,B,B4,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,N,2013,0


In [4]:
# Remove accidental index columns 
df_main = df_main.loc[:, ~df_main.columns.str.startswith("Unnamed")].copy()

# Tidy the term strings in place
df_main["term"] = df_main["term"].astype(str).str.strip()

print("Rows:", df_main.shape[0])
print("Columns:", df_main.shape[1])

display(df_main.head())

Rows: 1176374
Columns: 143


,id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,emp_title,...,payment_plan_start_date,hardship_length,hardship_dpd,hardship_loan_status,orig_projected_additional_accrued_interest,hardship_payoff_balance_amount,hardship_last_payment_amount,debt_settlement_flag,issue_year,target_bad_loan
0,10129454,12000.0,12000.0,12000.0,36 months,10.99%,392.81,B,B2,Project Manager,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,N,2013,0
1,10149488,4800.0,4800.0,4800.0,36 months,10.99%,157.13,B,B2,Surgical Technician,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,N,2013,0
2,10149342,27050.0,27050.0,27050.0,36 months,10.99%,885.46,B,B2,Team Leadern Customer Ops & Systems,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,N,2013,0
3,10148122,12000.0,12000.0,12000.0,36 months,7.62%,373.94,A,A3,Systems Engineer,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,N,2013,0
4,10129477,14000.0,14000.0,14000.0,36 months,12.85%,470.71,B,B4,Assistant Director - Human Resources,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,N,2013,0


## 2. Checking the Target and Time Coverage

In [5]:
print(f"Loaded modelling rows: {df_main.shape[0]:,}")
print(f"Loaded modelling features: {df_main.shape[1]}")
# Check the binary target balance
target_counts = df_main["target_bad_loan"].value_counts(dropna=False)
target_proportions = (df_main["target_bad_loan"].value_counts(normalize=True, dropna=False) * 100).round(2)

target_summary = pd.DataFrame({"count": target_counts,"percentage": target_proportions})

print("Target balance: 0 = Fully Paid, 1 = Charged Off/Default")
display(target_summary)

Loaded modelling rows: 1,176,374
Loaded modelling features: 143
Target balance: 0 = Fully Paid, 1 = Charged Off/Default


,count,percentage
target_bad_loan,,
0,954805,81.17
1,221569,18.83


In [6]:
# Confirm the dataset covers the intended period
df_main = df_main.copy()

df_main["issue_d"] = pd.to_datetime(df_main["issue_d"], errors="coerce")
df_main["issue_year"] = df_main["issue_d"].dt.year.astype("Int64")

print("Min issue date:", df_main["issue_d"].min())
print("Max issue date:", df_main["issue_d"].max())

display(df_main["issue_year"].value_counts().sort_index())

Min issue date: 2013-01-01 00:00:00
Max issue date: 2016-12-01 00:00:00


issue_year
2013    134814
2014    235619
2015    402837
2016    403104
Name: count, dtype: Int64

## 3. Chronological Study Design

- Historical base period: 2013
- Out-of-time simulation period: 2014–2016

2013 is used to train the champion and to define the monitoring baselines. The 2014–2016 loans are held back entirely and replayed month by month later.

## 4. Reviewing Columns and Assigning Feature Roles

Rather than dropping columns ad hoc, every column is given an explicit role: identifier, design filter, time, target, leakage, drop, or feature. Keeping this in one table makes it easy to show why each variable was included or excluded, and to check that nothing was accidentally left in two groups at once.

In [7]:
# Look at every column before assigning roles
all_columns = df_main.columns.tolist()

print(f"Total columns found: {len(all_columns)}")
print("\nColumn names:")

for i in range(0, len(all_columns), 5):
    print(all_columns[i:i + 5])

Total columns found: 143

Column names:
['id', 'loan_amnt', 'funded_amnt', 'funded_amnt_inv', 'term']
['int_rate', 'installment', 'grade', 'sub_grade', 'emp_title']
['emp_length', 'home_ownership', 'annual_inc', 'verification_status', 'issue_d']
['loan_status', 'pymnt_plan', 'url', 'purpose', 'title']
['zip_code', 'addr_state', 'dti', 'delinq_2yrs', 'earliest_cr_line']
['fico_range_low', 'fico_range_high', 'inq_last_6mths', 'mths_since_last_delinq', 'mths_since_last_record']
['open_acc', 'pub_rec', 'revol_bal', 'revol_util', 'total_acc']
['initial_list_status', 'out_prncp', 'out_prncp_inv', 'total_pymnt', 'total_pymnt_inv']
['total_rec_prncp', 'total_rec_int', 'total_rec_late_fee', 'recoveries', 'collection_recovery_fee']
['last_pymnt_d', 'last_pymnt_amnt', 'next_pymnt_d', 'last_credit_pull_d', 'last_fico_range_high']
['last_fico_range_low', 'collections_12_mths_ex_med', 'mths_since_last_major_derog', 'policy_code', 'application_type']
['annual_inc_joint', 'dti_joint', 'verification_st

In [8]:
# Audit dtypes, missingness and cardinality
column_audit = pd.DataFrame({"column": df_main.columns,"dtype": df_main.dtypes.astype(str).values,
                             "missing_count": df_main.isna().sum().values,
                             "missing_pct": (df_main.isna().mean() * 100).round(3).values,
                             "nunique": df_main.nunique(dropna=False).values})

column_audit = column_audit.sort_values(["missing_pct", "nunique"], ascending=[False, True])

print("Total columns:", df_main.shape[1])
display(column_audit)

Total columns: 143


,column,dtype,missing_count,missing_pct,nunique
113,revol_bal_joint,float64,1176374,100.0,1
114,sec_app_fico_range_low,float64,1176374,100.0,1
115,sec_app_fico_range_high,float64,1176374,100.0,1
116,sec_app_earliest_cr_line,float64,1176374,100.0,1
117,sec_app_inq_last_6mths,float64,1176374,100.0,1
...,...,...,...,...,...
46,last_pymnt_amnt,float64,0,0.0,574476
39,total_pymnt_inv,float64,0,0.0,894986
38,total_pymnt,float64,0,0.0,1087452
0,id,int64,0,0.0,1176374


In [9]:
# Check whether any column names are duplicated
duplicated_columns = df_main.columns[df_main.columns.duplicated()].tolist()

print("Duplicated columns:", len(duplicated_columns))

if len(duplicated_columns) > 0:
    print(duplicated_columns)

Duplicated columns: 0


In [10]:
# Identify columns with only one unique value
constant_columns = column_audit[column_audit["nunique"] <= 1]

print("Constant columns:", len(constant_columns))
display(constant_columns)

Constant columns: 14


,column,dtype,missing_count,missing_pct,nunique
113,revol_bal_joint,float64,1176374,100.0,1
114,sec_app_fico_range_low,float64,1176374,100.0,1
115,sec_app_fico_range_high,float64,1176374,100.0,1
116,sec_app_earliest_cr_line,float64,1176374,100.0,1
117,sec_app_inq_last_6mths,float64,1176374,100.0,1
118,sec_app_mort_acc,float64,1176374,100.0,1
119,sec_app_open_acc,float64,1176374,100.0,1
120,sec_app_revol_util,float64,1176374,100.0,1
121,sec_app_open_act_il,float64,1176374,100.0,1
122,sec_app_num_rev_accts,float64,1176374,100.0,1


In [11]:
# Missingness can shift over time, so check by year
missing_by_year_all = (df_main.groupby("issue_year").apply(lambda x: x.isna().mean() * 100).T)

missing_by_year_all["max_missing_pct"] = missing_by_year_all.max(axis=1)
missing_by_year_all = missing_by_year_all.sort_values("max_missing_pct", ascending=False)

with pd.option_context("display.max_rows", None):
    display(missing_by_year_all.drop(columns="max_missing_pct").round(2))

issue_year,2013,2014,2015,2016
dti_joint,100.00,100.00,99.89,98.11
annual_inc_joint,100.00,100.00,99.89,98.11
open_acc_6m,100.00,100.00,95.03,0.01
inq_fi,100.00,100.00,95.03,0.01
all_util,100.00,100.00,95.03,0.02
max_bal_bc,100.00,100.00,95.03,0.01
open_rv_24m,100.00,100.00,95.03,0.01
open_rv_12m,100.00,100.00,95.03,0.01
il_util,100.00,100.00,95.67,13.32
total_bal_il,100.00,100.00,95.03,0.01


The lists below are written out by hand so the reasoning stays visible. `term` is treated as a design filter rather than a predictor: it defines the cohort and is useful for later checks, but letting the model use it risks picking up maturity effects instead of borrower risk.

In [12]:
# One row per column, defaulting to review
column_roles = pd.DataFrame({"column": df_main.columns})
column_roles["role"] = "review"
column_roles["reason"] = ""

# Identifier columns are not useful for modelling
id_cols = ["id", "url"]

# Used to define the cohort, not to predict
# Term is kept for cohort checks only, never as a predictor
design_filter_cols = ["term"]

# Kept for splitting and feature engineering
time_cols = ["issue_d", "issue_year", "earliest_cr_line"]

# Target-related columns are excluded from the predictors
target_cols = ["loan_status", "target_bad_loan"]

# Anything known only after origination is leakage
leakage_cols = ["out_prncp", "out_prncp_inv", "total_pymnt", "total_pymnt_inv",
                "total_rec_prncp", "total_rec_int", "total_rec_late_fee",
                "recoveries", "collection_recovery_fee", "last_pymnt_d",
                "last_pymnt_amnt", "next_pymnt_d", "last_credit_pull_d",
                "last_fico_range_high", "last_fico_range_low",
                "hardship_flag", "hardship_type", "hardship_reason", "hardship_status",
                "deferral_term", "hardship_amount", "hardship_start_date",
                "hardship_end_date", "payment_plan_start_date", "hardship_length",
                "hardship_dpd", "hardship_loan_status",
                "orig_projected_additional_accrued_interest",
                "hardship_payoff_balance_amount", "hardship_last_payment_amount",
                "debt_settlement_flag", "pymnt_plan"]

# Excluded: availability changes over time
structural_drop_cols = ["il_util", "mths_since_rcnt_il", "all_util", "open_acc_6m",
                        "inq_last_12m", "total_cu_tl", "open_il_12m", "open_rv_12m",
                        "inq_fi", "open_il_24m", "open_rv_24m", "open_act_il",
                        "max_bal_bc", "total_bal_il"]

# Joint and secondary-applicant fields have excessive missingness for the main baseline
secondary_applicant_drop_cols = ["revol_bal_joint", "sec_app_fico_range_low", "sec_app_fico_range_high",
                                 "sec_app_earliest_cr_line", "sec_app_inq_last_6mths", "sec_app_mort_acc",
                                 "sec_app_open_acc", "sec_app_revol_util", "sec_app_open_act_il",
                                 "sec_app_num_rev_accts", "sec_app_chargeoff_within_12_mths",
                                 "sec_app_collections_12_mths_ex_med", "annual_inc_joint", "dti_joint",
                                 "verification_status_joint"]

# Other sparse, unsuitable, or unstable variables are excluded from the main baseline
other_drop_cols = ["mths_since_last_record", "mths_since_last_major_derog",
                   "mths_since_recent_bc_dlq", "mths_since_recent_revol_delinq",
                   "funded_amnt", "funded_amnt_inv",
                   "policy_code", "title", "emp_title", "zip_code", "addr_state", "application_type"]

all_drop_cols = structural_drop_cols + secondary_applicant_drop_cols + other_drop_cols


In [13]:
# Candidate origination-time features for the main modelling table
keep_candidate_cols = ["loan_amnt", "int_rate", "installment", "grade", "sub_grade",
                       "emp_length", "home_ownership", "annual_inc", "verification_status",
                       "purpose", "dti", "delinq_2yrs", "fico_range_low", "fico_range_high",
                       "inq_last_6mths", "mths_since_last_delinq", "open_acc", "pub_rec",
                       "revol_bal", "revol_util", "total_acc", "initial_list_status",
                       "collections_12_mths_ex_med", "acc_now_delinq",
                       "tot_coll_amt", "tot_cur_bal", "total_rev_hi_lim",
                       "acc_open_past_24mths", "avg_cur_bal", "bc_open_to_buy", "bc_util",
                       "chargeoff_within_12_mths", "delinq_amnt", "mo_sin_old_il_acct",
                       "mo_sin_old_rev_tl_op", "mo_sin_rcnt_rev_tl_op", "mo_sin_rcnt_tl",
                       "mort_acc", "mths_since_recent_bc", "mths_since_recent_inq",
                       "num_accts_ever_120_pd", "num_actv_bc_tl", "num_actv_rev_tl",
                       "num_bc_sats", "num_bc_tl", "num_il_tl", "num_op_rev_tl",
                       "num_rev_accts", "num_rev_tl_bal_gt_0", "num_sats",
                       "num_tl_120dpd_2m", "num_tl_30dpd", "num_tl_90g_dpd_24m",
                       "num_tl_op_past_12m", "pct_tl_nvr_dlq", "percent_bc_gt_75",
                       "pub_rec_bankruptcies", "tax_liens", "tot_hi_cred_lim",
                       "total_bal_ex_mort", "total_bc_limit", "total_il_high_credit_limit"]

In [14]:
# Catch typos in the manual role lists
role_groups = {"id": id_cols,"design_filter": design_filter_cols,"time": time_cols,"target": target_cols,"leakage": leakage_cols,
               "drop": all_drop_cols,"feature": keep_candidate_cols}

for role_name, cols in role_groups.items():
    missing_cols = sorted(set(cols) - set(df_main.columns))
    
    if len(missing_cols) > 0:
        print(f"{role_name}: missing from dataframe")
        print(missing_cols)

# Check whether any column has accidentally been placed in more than one role group
role_check = pd.DataFrame([(col, role_name) for role_name, cols in role_groups.items() for col in cols],columns=["column", "role"])

duplicate_role_cols = role_check[role_check.duplicated("column", keep=False)].sort_values("column")

print("Columns assigned to more than one role:", duplicate_role_cols["column"].nunique())

if len(duplicate_role_cols) > 0:
    display(duplicate_role_cols)

Columns assigned to more than one role: 0


In [15]:
# Apply the roles
column_roles.loc[column_roles["column"].isin(id_cols), ["role", "reason"]] = ["id", "Identifier column"]

column_roles.loc[column_roles["column"].isin(design_filter_cols),["role", "reason"]] = ["design_filter","Used to define the selected 36- and 60-month modelling cohort"]

column_roles.loc[column_roles["column"].isin(time_cols), ["role", "reason"]] = ["time", "Time-related column"]

column_roles.loc[column_roles["column"].isin(target_cols), ["role", "reason"]] = ["target", "Target or target-related column"]

column_roles.loc[column_roles["column"].isin(leakage_cols), ["role", "reason"]] = ["leakage", "Post-origination or outcome-related leakage"]

column_roles.loc[column_roles["column"].isin(all_drop_cols), ["role", "reason"]] = ["drop", "Failed missingness, sparsity, redundancy, or stability checks"]

column_roles.loc[column_roles["column"].isin(keep_candidate_cols), ["role", "reason"]] = ["feature", "Candidate origination-time modelling feature"]

# Add summary information for each column
column_roles["dtype"] = column_roles["column"].map(df_main.dtypes.astype(str))
column_roles["missing_count"] = column_roles["column"].map(df_main.isna().sum())
column_roles["missing_pct"] = column_roles["column"].map((df_main.isna().mean() * 100).round(3))
column_roles["nunique"] = column_roles["column"].map(df_main.nunique(dropna=False))

In [16]:
# Check whether any columns still need a role
remaining_review_cols = column_roles[column_roles["role"] == "review"].sort_values(["missing_pct", "nunique"], ascending=[False, True])

print(f"Columns still needing a role assigned: {len(remaining_review_cols)}")

if len(remaining_review_cols) > 0:
    display(remaining_review_cols)

Columns still needing a role assigned: 0


In [17]:
# Summarise how many columns are assigned to each role
role_summary = column_roles["role"].value_counts()

display(role_summary)

role
feature          62
drop             41
leakage          32
time              3
id                2
target            2
design_filter     1
Name: count, dtype: int64

In [18]:
# Display the columns inside each role group
for role_name in sorted(column_roles["role"].unique()):
    role_columns = column_roles.loc[column_roles["role"] == role_name, "column"].tolist()
    print(f"\n--- {role_name.upper()} ({len(role_columns)}) ---")
    print(role_columns)


--- DESIGN_FILTER (1) ---
['term']

--- DROP (41) ---
['funded_amnt', 'funded_amnt_inv', 'emp_title', 'title', 'zip_code', 'addr_state', 'mths_since_last_record', 'mths_since_last_major_derog', 'policy_code', 'application_type', 'annual_inc_joint', 'dti_joint', 'verification_status_joint', 'open_acc_6m', 'open_act_il', 'open_il_12m', 'open_il_24m', 'mths_since_rcnt_il', 'total_bal_il', 'il_util', 'open_rv_12m', 'open_rv_24m', 'max_bal_bc', 'all_util', 'inq_fi', 'total_cu_tl', 'inq_last_12m', 'mths_since_recent_bc_dlq', 'mths_since_recent_revol_delinq', 'revol_bal_joint', 'sec_app_fico_range_low', 'sec_app_fico_range_high', 'sec_app_earliest_cr_line', 'sec_app_inq_last_6mths', 'sec_app_mort_acc', 'sec_app_open_acc', 'sec_app_revol_util', 'sec_app_open_act_il', 'sec_app_num_rev_accts', 'sec_app_chargeoff_within_12_mths', 'sec_app_collections_12_mths_ex_med']

--- FEATURE (62) ---
['loan_amnt', 'int_rate', 'installment', 'grade', 'sub_grade', 'emp_length', 'home_ownership', 'annual_inc',

In [19]:
# Every column must have exactly one role
role_coverage_check = pd.DataFrame({"total_dataframe_columns": [df_main.shape[1]],
                                    "total_role_rows": [len(column_roles)],
                                    "unique_role_columns": [column_roles["column"].nunique()],
                                    "unassigned_columns": [(column_roles["role"] == "review").sum()]})

display(role_coverage_check)

,total_dataframe_columns,total_role_rows,unique_role_columns,unassigned_columns
0,143,143,143,0


`int_rate`,`sub_grade` and `grade` are not purely borrower attributes. Lending Club assigns the grade and sets the rate. I kept them because they are available at origination and strongly predictive, so they do not leak outcome information.

This matters later for drift monitoring. A shift in these variables can mean Lending Club changed its pricing or underwriting rather than that borrowers changed, so drift here needs investigating before it is read as a model problem. 

## 6. Engineering and Cleaning Predictor Variables

This section converts variables into modelling-ready formats and adds two engineered features, `months_credit_history` and `fico_avg`. Only transformations that use information available at origination are applied here, before the chronological split.

In [20]:
# Make a clean copy before adding engineered columns
df_main = df_main.copy()

# Convert date columns
df_main["issue_d"] = pd.to_datetime(df_main["issue_d"], errors="coerce")
df_main["earliest_cr_line"] = pd.to_datetime(df_main["earliest_cr_line"].astype(str).str.strip(),format="%b-%Y",errors="coerce")

# Build both at once to avoid fragmentation warnings
engineered_features = pd.DataFrame(index=df_main.index)

engineered_features["months_credit_history"] = ((df_main["issue_d"].dt.year - df_main["earliest_cr_line"].dt.year) * 12 +
                                                (df_main["issue_d"].dt.month - df_main["earliest_cr_line"].dt.month))

engineered_features["fico_avg"] = (df_main["fico_range_low"] + df_main["fico_range_high"]) / 2

df_main = pd.concat([df_main.drop(columns=engineered_features.columns, errors="ignore"), engineered_features],axis=1).copy()

date_check = pd.DataFrame({"missing_count": df_main[["issue_d", "earliest_cr_line"]].isna().sum(),
                           "missing_pct": (df_main[["issue_d", "earliest_cr_line"]].isna().mean() * 100).round(3)})

print("Date conversion check:")
display(date_check)

print("\nEngineered feature check:")
display(df_main[["issue_d", "earliest_cr_line", "months_credit_history", "fico_avg"]].head())

Date conversion check:


,missing_count,missing_pct
issue_d,0,0.0
earliest_cr_line,0,0.0



Engineered feature check:


,issue_d,earliest_cr_line,months_credit_history,fico_avg
0,2013-12-01,2009-12-01,48,722.0
1,2013-12-01,1995-08-01,220,757.0
2,2013-12-01,1986-10-01,326,732.0
3,2013-12-01,2003-09-01,123,707.0
4,2013-12-01,1988-06-01,306,672.0


In [21]:
# fico_avg replaces the two raw FICO bounds
raw_fico_cols = ["fico_range_low", "fico_range_high"]
new_feature_cols = ["months_credit_history", "fico_avg"]

column_roles.loc[column_roles["column"].isin(raw_fico_cols), ["role", "reason"]] = ["drop", "Combined into engineered fico_avg feature"]

# Keeps the tracker correct if this cell is re-run
column_roles = column_roles[~column_roles["column"].isin(new_feature_cols)].copy()

new_feature_rows = pd.DataFrame({"column": new_feature_cols,"role": "feature","reason": "Engineered origination-time feature"})

column_roles = pd.concat([column_roles, new_feature_rows], ignore_index=True)

# Refresh summary information
column_roles["dtype"] = column_roles["column"].map(df_main.dtypes.astype(str))
column_roles["missing_count"] = column_roles["column"].map(df_main.isna().sum())
column_roles["missing_pct"] = column_roles["column"].map((df_main.isna().mean() * 100).round(3))
column_roles["nunique"] = column_roles["column"].map(df_main.nunique(dropna=False))

display(column_roles[column_roles["column"].isin(raw_fico_cols + new_feature_cols)])

,column,role,reason,dtype,missing_count,missing_pct,nunique
25,fico_range_low,drop,Combined into engineered fico_avg feature,float64,0,0.0,38
26,fico_range_high,drop,Combined into engineered fico_avg feature,float64,0,0.0,38
143,months_credit_history,feature,Engineered origination-time feature,int32,0,0.0,720
144,fico_avg,feature,Engineered origination-time feature,float64,0,0.0,38


In [22]:
# Rebuild the feature list from the role tracker

final_feature_cols = (column_roles.loc[column_roles["role"] == "feature", "column"].tolist())

# Keep only features that are currently present in df_main
final_feature_cols = [col for col in final_feature_cols if col in df_main.columns]

# Remove accidental duplicates while preserving order
final_feature_cols = list(dict.fromkeys(final_feature_cols))

print("Final feature count:", len(final_feature_cols))
print(final_feature_cols)

Final feature count: 62
['loan_amnt', 'int_rate', 'installment', 'grade', 'sub_grade', 'emp_length', 'home_ownership', 'annual_inc', 'verification_status', 'purpose', 'dti', 'delinq_2yrs', 'inq_last_6mths', 'mths_since_last_delinq', 'open_acc', 'pub_rec', 'revol_bal', 'revol_util', 'total_acc', 'initial_list_status', 'collections_12_mths_ex_med', 'acc_now_delinq', 'tot_coll_amt', 'tot_cur_bal', 'total_rev_hi_lim', 'acc_open_past_24mths', 'avg_cur_bal', 'bc_open_to_buy', 'bc_util', 'chargeoff_within_12_mths', 'delinq_amnt', 'mo_sin_old_il_acct', 'mo_sin_old_rev_tl_op', 'mo_sin_rcnt_rev_tl_op', 'mo_sin_rcnt_tl', 'mort_acc', 'mths_since_recent_bc', 'mths_since_recent_inq', 'num_accts_ever_120_pd', 'num_actv_bc_tl', 'num_actv_rev_tl', 'num_bc_sats', 'num_bc_tl', 'num_il_tl', 'num_op_rev_tl', 'num_rev_accts', 'num_rev_tl_bal_gt_0', 'num_sats', 'num_tl_120dpd_2m', 'num_tl_30dpd', 'num_tl_90g_dpd_24m', 'num_tl_op_past_12m', 'pct_tl_nvr_dlq', 'percent_bc_gt_75', 'pub_rec_bankruptcies', 'tax_li

In [23]:
# Check missingness by issue year for the final feature set
missing_by_year = (df_main.groupby("issue_year")[final_feature_cols].apply(lambda x: x.isna().mean() * 100).T)

missing_by_year["max_missing_pct"] = missing_by_year.max(axis=1)
missing_by_year = missing_by_year.sort_values("max_missing_pct", ascending=False)

print("Missingness % by year for final features:")

with pd.option_context("display.max_rows", None):
    display(missing_by_year.drop(columns="max_missing_pct").round(2))

Missingness % by year for final features:


issue_year,2013,2014,2015,2016
mths_since_last_delinq,56.50,49.18,48.39,47.11
mths_since_recent_inq,10.82,9.21,10.45,10.30
emp_length,4.42,5.10,5.71,6.65
num_tl_120dpd_2m,0.19,3.34,4.58,4.81
mo_sin_old_il_acct,3.68,3.04,2.94,2.75
bc_util,0.79,1.11,1.01,1.14
percent_bc_gt_75,0.75,1.08,1.02,1.10
bc_open_to_buy,0.75,1.04,0.95,1.09
mths_since_recent_bc,0.65,0.95,0.91,1.03
pct_tl_nvr_dlq,0.11,0.00,0.00,0.00


In [24]:
# Audit dtypes, missingness and ranges before cleaning
audit_rows = []

for col in final_feature_cols:
    series = df_main[col]
    dtype = str(series.dtype)
    missing_pct = round(series.isna().mean() * 100, 2)
    
    if pd.api.types.is_numeric_dtype(series):
        value_summary = f"Min: {series.min():,.2f} | Max: {series.max():,.2f}"
    else:
        unique_values = series.dropna().unique()
        n_unique = len(unique_values)
        
        if n_unique <= 15:
            value_summary = f"{n_unique} categories: {list(unique_values)}"
        else:
            value_summary = f"{n_unique} categories: {list(unique_values[:5])}..."
    
    audit_rows.append({"feature": col,
                       "dtype": dtype,
                       "missing_pct": missing_pct,
                       "value_summary": value_summary})

feature_audit = pd.DataFrame(audit_rows)

with pd.option_context("display.max_rows", None, "display.max_colwidth", None):
    display(feature_audit)

,feature,dtype,missing_pct,value_summary
0,loan_amnt,float64,0.00,"Min: 1,000.00 | Max: 40,000.00"
1,int_rate,str,0.00,"262 categories: [' 10.99%', ' 7.62%', ' 12.85%', ' 6.62%', ' 8.90%']..."
2,installment,float64,0.00,"Min: 4.93 | Max: 1,584.90"
3,grade,str,0.00,"7 categories: ['B', 'A', 'C', 'D', 'E', 'F', 'G']"
4,sub_grade,str,0.00,"35 categories: ['B2', 'A3', 'B4', 'A2', 'A5']..."
5,emp_length,str,5.76,"11 categories: ['4 years', '2 years', '10+ years', '3 years', '5 years', '6 years', '1 year', '7 years', '< 1 year', '9 years', '8 years']"
6,home_ownership,str,0.00,"4 categories: ['RENT', 'MORTGAGE', 'OWN', 'ANY']"
7,annual_inc,float64,0.00,"Min: 0.00 | Max: 9,573,072.00"
8,verification_status,str,0.00,"3 categories: ['Not Verified', 'Source Verified', 'Verified']"
9,purpose,str,0.00,"14 categories: ['debt_consolidation', 'home_improvement', 'other', 'credit_card', 'major_purchase', 'small_business', 'house', 'moving', 'medical', 'car', 'vacation', 'renewable_energy', 'wedding', 'educational']"


## 7. Cleaning Predictor Data Types

Several fields arrive as text and need converting before modelling. Percentages stored with a `%` sign, employment length as a string band, and the Lending Club grades. Grade and sub-grade are encoded ordinally because their order is meaningful, while the unordered categories are left as `category` dtype for XGBoost later

In [25]:
# Convert percentage-style columns into numeric percentage points

percentage_cols = ["int_rate", "revol_util"]

for col in percentage_cols:df_main[col] = (pd.to_numeric(df_main[col].astype(str)
    .str.replace("%", "", regex=False).str.strip(),errors="coerce"))

percentage_check = pd.DataFrame({"dtype": df_main[percentage_cols].dtypes.astype(str),
                                 "missing_pct": (df_main[percentage_cols].isna().mean() * 100).round(3),
                                 "min": df_main[percentage_cols].min(),
                                 "max": df_main[percentage_cols].max()})

print("Percentage column cleaning check:")
display(percentage_check)

Percentage column cleaning check:


,dtype,missing_pct,min,max
int_rate,float64,0.000,5.32,30.99
revol_util,float64,0.052,0.00,892.30


In [26]:
# Convert employment length into an ordered numeric variable
emp_length_mapping = {"< 1 year": 0,"1 year": 1,"2 years": 2,"3 years": 3,"4 years": 4,"5 years": 5,
                      "6 years": 6,"7 years": 7,"8 years": 8,"9 years": 9,"10+ years": 10}

print("Employment length values before cleaning:")
display(df_main["emp_length"].value_counts(dropna=False).sort_index())

df_main["emp_length"] = df_main["emp_length"].replace(emp_length_mapping)
df_main["emp_length"] = pd.to_numeric(df_main["emp_length"], errors="coerce")

emp_length_check = pd.DataFrame({"dtype": [str(df_main["emp_length"].dtype)],
                                 "missing_pct": [(df_main["emp_length"].isna().mean() * 100).round(3)],
                                 "min": [df_main["emp_length"].min()],
                                 "max": [df_main["emp_length"].max()]})

print("\nEmployment length cleaning check:")
display(emp_length_check)

Employment length values before cleaning:


emp_length
1 year        76107
10+ years    397143
2 years      104792
3 years       92972
4 years       67977
5 years       71320
6 years       53467
7 years       52081
8 years       56491
9 years       46298
< 1 year      89957
NaN           67769
Name: count, dtype: int64


Employment length cleaning check:


,dtype,missing_pct,min,max
0,float64,5.761,0.0,10.0


In [27]:
# Ordinally encode Lending Club grade and sub-grade
print("Grade values before encoding:")
display(df_main["grade"].value_counts(dropna=False).sort_index())

print("\nSub-grade values before encoding:")
display(df_main["sub_grade"].value_counts(dropna=False).sort_index().head(10))

grade_mapping = {"A": 1, "B": 2, "C": 3, "D": 4, "E": 5, "F": 6, "G": 7}
subgrade_order = [f"{grade}{number}" for grade in "ABCDEFG" for number in range(1, 6)]
subgrade_mapping = {subgrade: i + 1 for i, subgrade in enumerate(subgrade_order)}

df_main["grade"] = df_main["grade"].map(grade_mapping)
df_main["sub_grade"] = df_main["sub_grade"].map(subgrade_mapping)

grade_check = pd.DataFrame({"dtype": df_main[["grade", "sub_grade"]].dtypes.astype(str),
                            "missing_pct": (df_main[["grade", "sub_grade"]].isna().mean() * 100).round(3),
                            "min": df_main[["grade", "sub_grade"]].min(),
                            "max": df_main[["grade", "sub_grade"]].max()})

print("\nGrade encoding check:")
display(grade_check)

Grade values before encoding:


grade
A    195864
B    346676
C    338498
D    175973
E     84371
F     28075
G      6917
Name: count, dtype: int64


Sub-grade values before encoding:


sub_grade
A1    38795
A2    29918
A3    29788
A4    42643
A5    54720
B1    62927
B2    65307
B3    71216
B4    75176
B5    72050
Name: count, dtype: int64


Grade encoding check:


,dtype,missing_pct,min,max
grade,int64,0.0,1,7
sub_grade,int64,0.0,1,35


In [28]:
# Encode the binary initial listing status
print("Initial listing status before encoding:")
display(df_main["initial_list_status"].value_counts(dropna=False))

df_main["initial_list_status"] = df_main["initial_list_status"].map({"f": 0, "w": 1})

initial_status_check = pd.DataFrame({"dtype": [str(df_main["initial_list_status"].dtype)],
                                     "missing_pct": [(df_main["initial_list_status"].isna().mean() * 100).round(3)],
                                     "min": [df_main["initial_list_status"].min()],
                                     "max": [df_main["initial_list_status"].max()]})

print("\nInitial listing status encoding check:")
display(initial_status_check)

Initial listing status before encoding:


initial_list_status
w    718463
f    457911
Name: count, dtype: int64


Initial listing status encoding check:


,dtype,missing_pct,min,max
0,int64,0.0,0,1


In [29]:
# Values outside 0-100 are treated as missing
df_main["dti"] = pd.to_numeric(df_main["dti"], errors="coerce")

invalid_dti_count = (~df_main["dti"].between(0, 100)).sum()

print("Invalid DTI values outside 0–100:", invalid_dti_count)

df_main.loc[~df_main["dti"].between(0, 100), "dti"] = np.nan

dti_check = pd.DataFrame({"dtype": [str(df_main["dti"].dtype)],
                          "missing_pct": [(df_main["dti"].isna().mean() * 100).round(3)],
                          "min": [df_main["dti"].min()],
                          "max": [df_main["dti"].max()]})

print("\nDTI cleaning check:")
display(dti_check)

Invalid DTI values outside 0–100: 179

DTI cleaning check:


,dtype,missing_pct,min,max
0,float64,0.015,0.0,100.0


In [30]:
# Left unencoded so XGBoost handles them in Notebook 03
nominal_categorical_cols = ["home_ownership", "verification_status", "purpose"]

for col in nominal_categorical_cols:
    df_main[col] = df_main[col].astype("category")

categorical_check = pd.DataFrame({"dtype": df_main[nominal_categorical_cols].dtypes.astype(str),
                                  "missing_pct": (df_main[nominal_categorical_cols].isna().mean() * 100).round(3),
                                  "nunique": df_main[nominal_categorical_cols].nunique(dropna=False)})

print("Nominal categorical column check:")
display(categorical_check)

for col in nominal_categorical_cols:
    print(f"\n{col}:")
    print(df_main[col].cat.categories.tolist())

Nominal categorical column check:


,dtype,missing_pct,nunique
home_ownership,category,0.0,4
verification_status,category,0.0,3
purpose,category,0.0,14



home_ownership:
['ANY', 'MORTGAGE', 'OWN', 'RENT']

verification_status:
['Not Verified', 'Source Verified', 'Verified']

purpose:
['car', 'credit_card', 'debt_consolidation', 'educational', 'home_improvement', 'house', 'major_purchase', 'medical', 'moving', 'other', 'renewable_energy', 'small_business', 'vacation', 'wedding']


In [31]:
# Refresh the final feature list after type cleaning
final_feature_cols = (column_roles.loc[column_roles["role"] == "feature", "column"].tolist())

final_feature_cols = [col for col in final_feature_cols if col in df_main.columns]
final_feature_cols = list(dict.fromkeys(final_feature_cols))

print(f"Final feature count after type cleaning: {len(final_feature_cols)}")

Final feature count after type cleaning: 62


In [32]:
# Re-audit after type cleaning
audit_rows = []

for col in final_feature_cols:
    series = df_main[col]
    dtype = str(series.dtype)
    missing_pct = round(series.isna().mean() * 100, 2)
    
    if pd.api.types.is_numeric_dtype(series):
        value_summary = f"Min: {series.min():,.2f} | Max: {series.max():,.2f}"
    else:
        unique_values = series.dropna().unique()
        n_unique = len(unique_values)
        
        if n_unique <= 15:
            value_summary = f"{n_unique} categories: {list(unique_values)}"
        else:
            value_summary = f"{n_unique} categories: {list(unique_values[:5])}..."
    
    audit_rows.append({"feature": col,
                       "dtype": dtype,
                       "missing_pct": missing_pct,
                       "value_summary": value_summary})

feature_audit_cleaned = pd.DataFrame(audit_rows)

with pd.option_context("display.max_rows", None, "display.max_colwidth", None):
    display(feature_audit_cleaned)

,feature,dtype,missing_pct,value_summary
0,loan_amnt,float64,0.00,"Min: 1,000.00 | Max: 40,000.00"
1,int_rate,float64,0.00,Min: 5.32 | Max: 30.99
2,installment,float64,0.00,"Min: 4.93 | Max: 1,584.90"
3,grade,int64,0.00,Min: 1.00 | Max: 7.00
4,sub_grade,int64,0.00,Min: 1.00 | Max: 35.00
5,emp_length,float64,5.76,Min: 0.00 | Max: 10.00
6,home_ownership,category,0.00,"4 categories: ['RENT', 'MORTGAGE', 'OWN', 'ANY']"
7,annual_inc,float64,0.00,"Min: 0.00 | Max: 9,573,072.00"
8,verification_status,category,0.00,"3 categories: ['Not Verified', 'Source Verified', 'Verified']"
9,purpose,category,0.00,"14 categories: ['debt_consolidation', 'home_improvement', 'other', 'credit_card', 'major_purchase', 'small_business', 'house', 'moving', 'medical', 'car', 'vacation', 'renewable_energy', 'wedding', 'educational']"


In [33]:
# Check missingness by issue year after type cleaning
missing_by_year_cleaned = (df_main.groupby("issue_year")[final_feature_cols].apply(lambda x: x.isna().mean() * 100).T)

missing_by_year_cleaned["max_missing_pct"] = missing_by_year_cleaned.max(axis=1)
missing_by_year_cleaned = missing_by_year_cleaned.sort_values("max_missing_pct", ascending=False)

print("Missingness % by year after type cleaning:")

with pd.option_context("display.max_rows", None):
    display(missing_by_year_cleaned.drop(columns="max_missing_pct").round(2))

Missingness % by year after type cleaning:


issue_year,2013,2014,2015,2016
mths_since_last_delinq,56.50,49.18,48.39,47.11
mths_since_recent_inq,10.82,9.21,10.45,10.30
emp_length,4.42,5.10,5.71,6.65
num_tl_120dpd_2m,0.19,3.34,4.58,4.81
mo_sin_old_il_acct,3.68,3.04,2.94,2.75
bc_util,0.79,1.11,1.01,1.14
percent_bc_gt_75,0.75,1.08,1.02,1.10
bc_open_to_buy,0.75,1.04,0.95,1.09
mths_since_recent_bc,0.65,0.95,0.91,1.03
pct_tl_nvr_dlq,0.11,0.00,0.00,0.00


## 8. Missingness Handling Before the Chronological Split

Missing values are handled in two stages. Where missingness has a clear meaning, for example, no recorded delinquency. It is filled here with a semantic value, and a missingness flag is kept so the model can still see that the value was originally absent.

Everything else is deliberately left as `NaN`. Those columns are imputed after the split in Section 10, using medians learned from 2013 only, so no information from the simulation period leaks backwards.

In [34]:
# Missingness before any handling is applied
missing_overall = (df_main[final_feature_cols].isna().mean() * 100).sort_values(ascending=False)
missing_overall = missing_overall[missing_overall > 0]

print("Features with missing values before missingness handling:")
display(missing_overall.to_frame("missing_pct").round(2))

Features with missing values before missingness handling:


,missing_pct
mths_since_last_delinq,49.04
mths_since_recent_inq,10.19
emp_length,5.76
num_tl_120dpd_2m,3.91
mo_sin_old_il_acct,2.98
bc_util,1.05
percent_bc_gt_75,1.03
bc_open_to_buy,0.99
mths_since_recent_bc,0.93
revol_util,0.05


In [35]:
# Flag first, so the fill does not hide the original gap
flag_base_cols = ["mths_since_last_delinq",
                  "mths_since_recent_inq",
                  "num_tl_120dpd_2m",
                  "mo_sin_old_il_acct",
                  "bc_util",
                  "bc_open_to_buy",
                  "percent_bc_gt_75"]

flag_base_cols = [col for col in flag_base_cols if col in final_feature_cols]
flag_cols = [f"{col}_missing" for col in flag_base_cols]

missing_flags = pd.DataFrame({f"{col}_missing": df_main[col].isna().astype(int)
                              for col in flag_base_cols}, index=df_main.index)

df_main = pd.concat([df_main.drop(columns=flag_cols, errors="ignore"), missing_flags],axis=1).copy()

print("Missingness flags created:")
print(flag_cols)

flag_summary = df_main[flag_cols].mean().mul(100).round(2).to_frame("flag_rate_pct")
display(flag_summary)

Missingness flags created:
['mths_since_last_delinq_missing', 'mths_since_recent_inq_missing', 'num_tl_120dpd_2m_missing', 'mo_sin_old_il_acct_missing', 'bc_util_missing', 'bc_open_to_buy_missing', 'percent_bc_gt_75_missing']


,flag_rate_pct
mths_since_last_delinq_missing,49.04
mths_since_recent_inq_missing,10.19
num_tl_120dpd_2m_missing,3.91
mo_sin_old_il_acct_missing,2.98
bc_util_missing,1.05
bc_open_to_buy_missing,0.99
percent_bc_gt_75_missing,1.03


In [36]:
# Add missingness flags to the final feature list
for col in flag_cols:
    if col not in final_feature_cols:
        final_feature_cols.append(col)

# Register missingness flags in the column-role tracker
column_roles = column_roles[~column_roles["column"].isin(flag_cols)].copy()

flag_role_rows = pd.DataFrame({"column": flag_cols,"role": "feature",
                               "reason": "Missingness indicator created before imputation"})

column_roles = pd.concat([column_roles, flag_role_rows], ignore_index=True)

# Refresh column-role summary information
column_roles["dtype"] = column_roles["column"].map(df_main.dtypes.astype(str))
column_roles["missing_count"] = column_roles["column"].map(df_main.isna().sum())
column_roles["missing_pct"] = column_roles["column"].map((df_main.isna().mean() * 100).round(3))
column_roles["nunique"] = column_roles["column"].map(df_main.nunique(dropna=False))

print(f"Final feature count after adding missingness flags: {len(final_feature_cols)}")
display(column_roles[column_roles["column"].isin(flag_cols)])

Final feature count after adding missingness flags: 69


,column,role,reason,dtype,missing_count,missing_pct,nunique
145,mths_since_last_delinq_missing,feature,Missingness indicator created before imputation,int64,0,0.0,2
146,mths_since_recent_inq_missing,feature,Missingness indicator created before imputation,int64,0,0.0,2
147,num_tl_120dpd_2m_missing,feature,Missingness indicator created before imputation,int64,0,0.0,2
148,mo_sin_old_il_acct_missing,feature,Missingness indicator created before imputation,int64,0,0.0,2
149,bc_util_missing,feature,Missingness indicator created before imputation,int64,0,0.0,2
150,bc_open_to_buy_missing,feature,Missingness indicator created before imputation,int64,0,0.0,2
151,percent_bc_gt_75_missing,feature,Missingness indicator created before imputation,int64,0,0.0,2


In [37]:
# Fill only where missing has a clear meaning
semantic_fill_values = {"mths_since_last_delinq": 999,
                        "mths_since_recent_inq": 999,
                        "num_tl_120dpd_2m": 0,
                        "mo_sin_old_il_acct": 0,
                        "bc_util": 0,
                        "bc_open_to_buy": 0,
                        "percent_bc_gt_75": 0}

semantic_fill_values = {col: value for col, value in semantic_fill_values.items()
                        if col in final_feature_cols}

df_main = df_main.fillna(value=semantic_fill_values)

semantic_check = df_main[list(semantic_fill_values.keys())].isna().sum().to_frame("missing_count_after_fill")

print("Semantic imputation check:")
display(semantic_check)

Semantic imputation check:


,missing_count_after_fill
mths_since_last_delinq,0
mths_since_recent_inq,0
num_tl_120dpd_2m,0
mo_sin_old_il_acct,0
bc_util,0
bc_open_to_buy,0
percent_bc_gt_75,0


In [38]:
# Left as NaN for leakage-free imputation after the split
median_impute_later_cols = ["dti","inq_last_6mths","emp_length","mths_since_recent_bc","pct_tl_nvr_dlq","revol_util","avg_cur_bal","num_rev_accts"]

median_impute_later_cols = [col for col in median_impute_later_cols
    if col in final_feature_cols]

remaining_missing = (df_main[median_impute_later_cols].isna().mean() * 100).round(2)

print("Columns reserved for leakage-free median imputation later:")
display(remaining_missing.to_frame("missing_pct"))

Columns reserved for leakage-free median imputation later:


,missing_pct
dti,0.02
inq_last_6mths,0.00
emp_length,5.76
mths_since_recent_bc,0.93
pct_tl_nvr_dlq,0.01
revol_util,0.05
avg_cur_bal,0.00
num_rev_accts,0.00


In [39]:
# Final check after semantic missingness handling
missing_after_semantic = (df_main[final_feature_cols].isna().mean() * 100).sort_values(ascending=False)
missing_after_semantic = missing_after_semantic[missing_after_semantic > 0]

print("Remaining missingness after semantic handling:")
display(missing_after_semantic.to_frame("missing_pct").round(4))

Remaining missingness after semantic handling:


,missing_pct
emp_length,5.7608
mths_since_recent_bc,0.9287
revol_util,0.0516
dti,0.0152
pct_tl_nvr_dlq,0.0130
avg_cur_bal,0.0010
num_rev_accts,0.0001
inq_last_6mths,0.0001


In [40]:
# Check remaining missingness by issue year after semantic handling
missing_by_year_after = (df_main.groupby("issue_year")[missing_after_semantic.index].apply(lambda x: x.isna().mean() * 100).T)

print("Remaining missingness % by year after semantic handling:")
display(missing_by_year_after.round(2))

Remaining missingness % by year after semantic handling:


issue_year,2013,2014,2015,2016
emp_length,4.42,5.10,5.71,6.65
mths_since_recent_bc,0.65,0.95,0.91,1.03
revol_util,0.06,0.05,0.04,0.06
dti,0.00,0.00,0.00,0.04
pct_tl_nvr_dlq,0.11,0.00,0.00,0.00
avg_cur_bal,0.00,0.00,0.00,0.00
num_rev_accts,0.00,0.00,0.00,0.00
inq_last_6mths,0.00,0.00,0.00,0.00


In [41]:
# Confirm the final feature list is clean after adding missingness flags
print("Final feature count:", len(final_feature_cols))
print("Duplicate features:", len(final_feature_cols) - len(set(final_feature_cols)))

print("\nRaw FICO columns still in final features:")
print([col for col in ["fico_range_low", "fico_range_high"] if col in final_feature_cols])

print("\nEngineered FICO feature included:")
print("fico_avg" in final_feature_cols)

Final feature count: 69
Duplicate features: 0

Raw FICO columns still in final features:
[]

Engineered FICO feature included:
True


## 9. Chronological Split

The modelling table is split by issue year: 2013 becomes the historical baseline and 2014–2016 the simulation stream. An `issue_month` label is added so the simulation period can later be replayed one month at a time.

In [42]:
# All features must exist before building the table
missing_feature_cols = sorted(set(final_feature_cols) - set(df_main.columns))

print("Missing final feature columns:", len(missing_feature_cols))

if len(missing_feature_cols) > 0:
    print(missing_feature_cols)

print("Final feature count:", len(final_feature_cols))
print("Duplicate features:", len(final_feature_cols) - len(set(final_feature_cols)))

Missing final feature columns: 0
Final feature count: 69
Duplicate features: 0


In [43]:
# Build the modelling table using final features, target, and time fields
model_cols = final_feature_cols + ["target_bad_loan", "issue_d", "issue_year"]

model_df = df_main[model_cols].copy()

# Month label drives the monthly replay in Notebook 04
model_df["issue_month"] = model_df["issue_d"].dt.to_period("M").astype(str)

print("Model dataframe shape:", model_df.shape)
display(model_df.head())

Model dataframe shape: (1176374, 73)


,loan_amnt,int_rate,installment,grade,sub_grade,emp_length,home_ownership,annual_inc,verification_status,purpose,...,mths_since_recent_inq_missing,num_tl_120dpd_2m_missing,mo_sin_old_il_acct_missing,bc_util_missing,bc_open_to_buy_missing,percent_bc_gt_75_missing,target_bad_loan,issue_d,issue_year,issue_month
0,12000.0,10.99,392.81,2,7,4.0,RENT,60000.0,Not Verified,debt_consolidation,...,0,0,1,0,0,0,0,2013-12-01,2013,2013-12
1,4800.0,10.99,157.13,2,7,2.0,MORTGAGE,39600.0,Source Verified,home_improvement,...,0,0,0,0,0,0,0,2013-12-01,2013,2013-12
2,27050.0,10.99,885.46,2,7,10.0,OWN,55000.0,Verified,debt_consolidation,...,0,0,0,0,0,0,0,2013-12-01,2013,2013-12
3,12000.0,7.62,373.94,1,3,3.0,MORTGAGE,96500.0,Not Verified,debt_consolidation,...,0,0,0,0,0,0,0,2013-12-01,2013,2013-12
4,14000.0,12.85,470.71,2,9,4.0,RENT,88000.0,Not Verified,debt_consolidation,...,1,0,0,0,0,0,0,2013-12-01,2013,2013-12


In [44]:
# Check the time coverage of the modelling table
coverage_summary = pd.DataFrame({"min_issue_date": [model_df["issue_d"].min()],
                                 "max_issue_date": [model_df["issue_d"].max()],
                                 "rows": [len(model_df)],
                                 "features": [len(final_feature_cols)]})

display(coverage_summary)

print("Rows by issue year:")
display(model_df["issue_year"].value_counts().sort_index())

,min_issue_date,max_issue_date,rows,features
0,2013-01-01,2016-12-01,1176374,69


Rows by issue year:


issue_year
2013    134814
2014    235619
2015    402837
2016    403104
Name: count, dtype: Int64

In [45]:
# 2013 baseline, 2014-2016 held out of time
historical_df = model_df[model_df["issue_year"] == 2013].copy()
simulation_df = model_df[model_df["issue_year"].isin([2014, 2015, 2016])].copy()

# Keep both in date order
historical_df = historical_df.sort_values("issue_d").reset_index(drop=True)
simulation_df = simulation_df.sort_values("issue_d").reset_index(drop=True)

print("Historical rows:", f"{len(historical_df):,}")
print("Simulation rows:", f"{len(simulation_df):,}")

Historical rows: 134,814
Simulation rows: 1,041,560


In [46]:
# Summarise the chronological split
split_summary = pd.DataFrame({"split": ["historical_2013", "simulation_2014_2016"],
                              "rows": [len(historical_df), len(simulation_df)],
                              "start_date": [historical_df["issue_d"].min(), simulation_df["issue_d"].min()],
                              "end_date": [historical_df["issue_d"].max(), simulation_df["issue_d"].max()],
                              "bad_loan_rate": [historical_df["target_bad_loan"].mean().round(4),
                                                simulation_df["target_bad_loan"].mean().round(4)]})

display(split_summary)

,split,rows,start_date,end_date,bad_loan_rate
0,historical_2013,134814,2013-01-01,2013-12-01,0.1560
1,simulation_2014_2016,1041560,2014-01-01,2016-12-01,0.1925


In [47]:
# Check target balance in each chronological split
historical_balance = historical_df["target_bad_loan"].value_counts(normalize=True).mul(100).round(2)
simulation_balance = simulation_df["target_bad_loan"].value_counts(normalize=True).mul(100).round(2)

class_balance = pd.DataFrame({"historical_2013_pct": historical_balance,"simulation_2014_2016_pct": simulation_balance})

display(class_balance)

,historical_2013_pct,simulation_2014_2016_pct
target_bad_loan,,
0,84.4,80.75
1,15.6,19.25


In [48]:
# Check monthly sample sizes for the simulation stream
monthly_simulation_counts = simulation_df.groupby("issue_month").size().to_frame("rows")

print("Monthly simulation batch sizes:")
display(monthly_simulation_counts)

Monthly simulation batch sizes:


,rows
issue_month,
2014-01,15628
2014-02,15269
2014-03,16513
2014-04,19071
2014-05,19099
2014-06,17179
2014-07,29306
2014-08,18814
2014-09,10606


Monthly counts matter for the later work. Each of these months becomes one simulated deployment batch, so months with very few loans will give noisier drift and performance estimates than the larger ones.

In [49]:
# Check remaining missingness after splitting
historical_missing = historical_df[final_feature_cols].isna().sum()
simulation_missing = simulation_df[final_feature_cols].isna().sum()

historical_missing = historical_missing[historical_missing > 0].sort_values(ascending=False)
simulation_missing = simulation_missing[simulation_missing > 0].sort_values(ascending=False)

print("Remaining missing values in historical features:")
display(historical_missing.to_frame("missing_count"))

print("\nRemaining missing values in simulation features:")
display(simulation_missing.to_frame("missing_count"))

Remaining missing values in historical features:


,missing_count
emp_length,5962
mths_since_recent_bc,880
pct_tl_nvr_dlq,153
revol_util,78
avg_cur_bal,6



Remaining missing values in simulation features:


,missing_count
emp_length,61807
mths_since_recent_bc,10045
revol_util,529
dti,179
avg_cur_bal,6
inq_last_6mths,1
num_rev_accts,1


## 10. Leakage-Free Imputation and Outlier Capping

The remaining preprocessing happens after the split, on purpose. Medians and 99th-percentile caps are learned from the 2013 baseline and then applied to both datasets. Fitting them on the pooled data would let 2014–2016 information influence the historical period and the model trained on it.

In [50]:
# Define columns that still need leakage-free median imputation
median_impute_cols = ["dti","inq_last_6mths", "emp_length","mths_since_recent_bc","pct_tl_nvr_dlq","revol_util","avg_cur_bal","num_rev_accts"]

median_impute_cols = [col for col in median_impute_cols if col in final_feature_cols]

missing_before_imputation = pd.DataFrame({"historical_missing": historical_df[median_impute_cols].isna().sum(),"simulation_missing": simulation_df[median_impute_cols].isna().sum()})

print("Columns selected for leakage-free median imputation:")
display(missing_before_imputation)

Columns selected for leakage-free median imputation:


,historical_missing,simulation_missing
dti,0,179
inq_last_6mths,0,1
emp_length,5962,61807
mths_since_recent_bc,880,10045
pct_tl_nvr_dlq,153,0
revol_util,78,529
avg_cur_bal,6,6
num_rev_accts,0,1


In [51]:
# Medians come from 2013 only
historical_medians = historical_df[median_impute_cols].median()

print("Historical medians used for imputation:")
display(historical_medians.to_frame("historical_median"))

missing_medians = historical_medians[historical_medians.isna()]

if len(missing_medians) > 0:
    print("Warning: some columns do not have a valid historical median.")
    display(missing_medians)

Historical medians used for imputation:


,historical_median
dti,16.89
inq_last_6mths,0.00
emp_length,7.00
mths_since_recent_bc,15.00
pct_tl_nvr_dlq,100.00
revol_util,60.30
avg_cur_bal,7990.00
num_rev_accts,14.00


In [52]:
# Same 2013 medians applied to both
historical_df[median_impute_cols] = historical_df[median_impute_cols].fillna(historical_medians)
simulation_df[median_impute_cols] = simulation_df[median_impute_cols].fillna(historical_medians)

missing_after_imputation = pd.DataFrame({"historical_missing": historical_df[median_impute_cols].isna().sum(),
                                         "simulation_missing": simulation_df[median_impute_cols].isna().sum()})

print("Missing values after leakage-free median imputation:")
display(missing_after_imputation)

Missing values after leakage-free median imputation:


,historical_missing,simulation_missing
dti,0,0
inq_last_6mths,0,0
emp_length,0,0
mths_since_recent_bc,0,0
pct_tl_nvr_dlq,0,0
revol_util,0,0
avg_cur_bal,0,0
num_rev_accts,0,0


In [53]:
# Check whether any missing values remain in the full feature space
historical_missing_after = historical_df[final_feature_cols].isna().sum()
simulation_missing_after = simulation_df[final_feature_cols].isna().sum()

historical_missing_after = historical_missing_after[historical_missing_after > 0].sort_values(ascending=False)
simulation_missing_after = simulation_missing_after[simulation_missing_after > 0].sort_values(ascending=False)

print("Remaining missing values in historical features:")
display(historical_missing_after.to_frame("missing_count"))

print("\nRemaining missing values in simulation features:")
display(simulation_missing_after.to_frame("missing_count"))

print("\nTotal historical feature NaNs:", historical_df[final_feature_cols].isna().sum().sum())
print("Total simulation feature NaNs:", simulation_df[final_feature_cols].isna().sum().sum())

Remaining missing values in historical features:


,missing_count



Remaining missing values in simulation features:


,missing_count



Total historical feature NaNs: 0
Total simulation feature NaNs: 0


In [54]:
# Heavy-tailed columns capped at the upper tail only
cap_cols = ["annual_inc", "revol_bal", "revol_util",
            "tot_coll_amt", "tot_cur_bal", "total_rev_hi_lim",
            "avg_cur_bal", "bc_open_to_buy", "bc_util",
            "mo_sin_old_il_acct", "mo_sin_old_rev_tl_op", "mths_since_recent_bc",
            "tot_hi_cred_lim", "total_bal_ex_mort",
            "total_bc_limit", "total_il_high_credit_limit", "months_credit_history"]

cap_cols = [col for col in cap_cols if col in final_feature_cols]

print("Columns selected for upper-tail capping:")
print(cap_cols)

Columns selected for upper-tail capping:
['annual_inc', 'revol_bal', 'revol_util', 'tot_coll_amt', 'tot_cur_bal', 'total_rev_hi_lim', 'avg_cur_bal', 'bc_open_to_buy', 'bc_util', 'mo_sin_old_il_acct', 'mo_sin_old_rev_tl_op', 'mths_since_recent_bc', 'tot_hi_cred_lim', 'total_bal_ex_mort', 'total_bc_limit', 'total_il_high_credit_limit', 'months_credit_history']


In [55]:
# Caps learned from 2013 only
cap_limits = historical_df[cap_cols].quantile(0.99)

cap_summary_before = pd.DataFrame({"historical_max_before": historical_df[cap_cols].max(),
                                   "simulation_max_before": simulation_df[cap_cols].max(),
                                   "historical_p99_cap": cap_limits})

print("Capping limits learned from 2013 historical data:")
display(cap_summary_before.round(2))

Capping limits learned from 2013 historical data:


,historical_max_before,simulation_max_before,historical_p99_cap
annual_inc,6100000.0,9573072.0,230000.00
revol_bal,2568995.0,2904836.0,81078.54
revol_util,140.4,892.3,97.40
tot_coll_amt,88303.0,9152545.0,1505.87
tot_cur_bal,4772549.0,5445012.0,638477.81
total_rev_hi_lim,9999999.0,9999999.0,133587.00
avg_cur_bal,958084.0,752994.0,72473.70
bc_open_to_buy,278899.0,559912.0,62681.79
bc_util,339.6,318.2,101.10
mo_sin_old_il_acct,647.0,724.0,269.00


In [56]:
# Same caps applied to both
for col in cap_cols:
    historical_df[col] = historical_df[col].clip(upper=cap_limits[col])
    simulation_df[col] = simulation_df[col].clip(upper=cap_limits[col])

cap_summary_after = pd.DataFrame({"historical_max_after": historical_df[cap_cols].max(),
                                  "simulation_max_after": simulation_df[cap_cols].max(),
                                  "historical_p99_cap": cap_limits})

print("Feature maxima after capping:")
display(cap_summary_after.round(2))

Feature maxima after capping:


,historical_max_after,simulation_max_after,historical_p99_cap
annual_inc,230000.00,230000.00,230000.00
revol_bal,81078.54,81078.54,81078.54
revol_util,97.40,97.40,97.40
tot_coll_amt,1505.87,1505.87,1505.87
tot_cur_bal,638477.81,638477.81,638477.81
total_rev_hi_lim,133587.00,133587.00,133587.00
avg_cur_bal,72473.70,72473.70,72473.70
bc_open_to_buy,62681.79,62681.79,62681.79
bc_util,101.10,101.10,101.10
mo_sin_old_il_acct,269.00,269.00,269.00


In [57]:
# Final shape, missingness, and target-rate checks after imputation and capping
final_split_check = pd.DataFrame({"dataset": ["historical_2013", "simulation_2014_2016"],
                                  "rows": [len(historical_df), len(simulation_df)],
                                  "columns": [historical_df.shape[1], simulation_df.shape[1]],
                                  "feature_count": [len(final_feature_cols), len(final_feature_cols)],
                                  "feature_nans": [historical_df[final_feature_cols].isna().sum().sum(),
                                                   simulation_df[final_feature_cols].isna().sum().sum()],
                                  "bad_loan_rate": [historical_df["target_bad_loan"].mean().round(4),
                                                    simulation_df["target_bad_loan"].mean().round(4)]})

display(final_split_check)

,dataset,rows,columns,feature_count,feature_nans,bad_loan_rate
0,historical_2013,134814,73,69,0,0.1560
1,simulation_2014_2016,1041560,73,69,0,0.1925


In [58]:
# These are the categoricals passed to Notebook 03
non_numeric_features = [col for col in final_feature_cols
                        if not pd.api.types.is_numeric_dtype(historical_df[col])]

print("Non-numeric final features:")
print(non_numeric_features)

for col in non_numeric_features:
    print(f"\n{col}:")
    print(historical_df[col].astype(str).value_counts(dropna=False).head(10))

Non-numeric final features:
['home_ownership', 'verification_status', 'purpose']

home_ownership:
home_ownership
MORTGAGE    72065
RENT        51498
OWN         11251
Name: count, dtype: int64

verification_status:
verification_status
Verified           66143
Not Verified       38959
Source Verified    29712
Name: count, dtype: int64

purpose:
purpose
debt_consolidation    80640
credit_card           32805
home_improvement       7404
other                  5843
major_purchase         2299
small_business         1359
car                    1050
medical                 889
house                   675
moving                  639
Name: count, dtype: int64


## 11. Saving the Processed Chronological Splits

This section saves the cleaned 2013 baseline, the simulation stream, and the preprocessing metadata: the feature list, the categorical columns, and the medians and caps learned from 2013. The categoricals are saved unencoded so encoding can be fitted inside the modelling pipeline using training data only.

In [59]:
# Save the splits and preprocessing metadata for Notebook 03
historical_path = "../data/processed/historical_2013_clean.parquet"
simulation_path = "../data/processed/simulation_2014_2016_clean.parquet"

features_path = "../data/processed/final_feature_cols.csv"
categorical_path = "../data/processed/categorical_feature_cols.csv"
median_path = "../data/processed/historical_imputation_medians.csv"
cap_path = "../data/processed/historical_capping_limits.csv"
split_summary_path = "../data/processed/split_summary.csv"

historical_df.to_parquet(historical_path, index=False)
simulation_df.to_parquet(simulation_path, index=False)

pd.Series(final_feature_cols, name="feature").to_csv(features_path, index=False)
pd.Series(non_numeric_features, name="categorical_feature").to_csv(categorical_path, index=False)
historical_medians.to_csv(median_path, header=["historical_median"])
cap_limits.to_csv(cap_path, header=["historical_p99_cap"])
split_summary.to_csv(split_summary_path, index=False)

print("Saved Notebook 02 outputs:")
print(historical_path, historical_df.shape)
print(simulation_path, simulation_df.shape)
print(features_path, len(final_feature_cols))
print(categorical_path, len(non_numeric_features))
print(median_path)
print(cap_path)
print(split_summary_path)

Saved Notebook 02 outputs:
../data/processed/historical_2013_clean.parquet (134814, 73)
../data/processed/simulation_2014_2016_clean.parquet (1041560, 73)
../data/processed/final_feature_cols.csv 69
../data/processed/categorical_feature_cols.csv 3
../data/processed/historical_imputation_medians.csv
../data/processed/historical_capping_limits.csv
../data/processed/split_summary.csv
